In [59]:
import numpy as np
import pandas as pd
import pyvinecopulib as pv

import json
import scipy.stats

from utils.constants import *
from utils.logging import LOGGER
from utils.datagen import load_s3_data_as_df, load_local_data_as_df

from generative_models.data_synthesiser_utils.datatypes.FloatAttribute import FloatAttribute
from generative_models.data_synthesiser_utils.datatypes.IntegerAttribute import IntegerAttribute
from generative_models.data_synthesiser_utils.datatypes.StringAttribute import StringAttribute

from generative_models.generative_model import GenerativeModel

from utils.constants import *
from utils.logging import LOGGER

import subprocess
import os
import json
import tempfile

from datetime import datetime
from pathlib import Path

CVINE_SENSITIVE_SCRIPT = Path("__file__").resolve().parent / "generative_models" / "run_cvinecop_sensitive_estimation.R"

In [60]:
class DataDescriber(object):
    def __init__(self, metadata, histogram_bins, infer_ranges=False):
        self.metadata = metadata
        self.histogram_bins = histogram_bins
        self.infer_ranges = infer_ranges

        self.attr_dict = None
        self.attr_names = None

    def describe(self, df):
        self.attr_names = self._get_attr_names()
        self.attr_dict = self._represent_input_dataset_by_columns(df)

        for col, Attribute in self.attr_dict.items():
            Attribute.infer_distribution()

    def _get_attr_names(self):
        return [c for c in self.metadata.keys()]

    def _represent_input_dataset_by_columns(self, df):
        attr_dict = {}

        for col, cdict in self.metadata.items():
            coltype = cdict['type']

            paras = (col, df[col], self.histogram_bins)
            if coltype in NUMERICAL:
                if coltype == FLOAT:
                    Attribute = FloatAttribute(*paras)
                else:
                    Attribute = IntegerAttribute(*paras)

                if self.infer_ranges:
                    cmin, cmax = min(df[col]), max(df[col])
                else:
                    cmin, cmax = cdict['min'], cdict['max']

                Attribute.set_domain(domain=(cmin, cmax))

            elif coltype in STRINGS:
                Attribute = StringAttribute(*paras)
                if self.infer_ranges:
                    ccats = list(df[col].unique())
                else:
                    ccats = cdict['categories']

                Attribute.set_domain(domain=ccats)

            else:
                raise Exception(f'The DataType of {col} is unknown.')

            attr_dict[col] = Attribute

        return attr_dict

In [61]:
class CvineSensitive(GenerativeModel):
    def __init__(self, 
                 metadata, 
                 family_set = "parametric",
                 sensitive = None,
                 lmbda = 0,
                 trunc_lvl = None, 
                 histogram_bins = 45, 
                 infer_ranges = False, 
                 multiprocess = True):
        """A C-vine copula model putting penalties on sensitive parameters during estimation"""
        self.metadata = self._read_meta(metadata)
        self.family_set = family_set
        self.sensitive = sensitive
        self.lmbda = lmbda 
        self.histogram_bins = histogram_bins
        self.multiprocess = bool(multiprocess)
        self.infer_ranges = bool(infer_ranges)

        self.d = len(metadata['columns'])
        self.trunc_lvl = trunc_lvl if trunc_lvl is not None else self.d - 1
        self.var_types = ['c' if metadata['columns'][i]['type'] == 'Float' else 'd' for i in range(self.d)]

        self.datatype = pd.DataFrame
        self.__name__ = 'Cvine'
        self.DataDescriber = None
        self.trained = False

        self.vc = None

    def fit(self, data):
        """Fit model..."""
        LOGGER.debug(f'Start fitting CVineSensitive({self.lmbda}) model to data of shape {data.shape}...')

        # Fit model by calling R-script (add more documentation ASAP)
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp = Path(tmpdir)
            data_path, params_path, model_path = tmp / "data.csv", tmp / "params.json", tmp / "model.json"

            # Export data to temporary csv file
            data.to_csv(data_path, index=False)

            # Export parameters to temporary json file
            params = {"var_types":self.var_types,
                      "family_set":self.family_set,
                      "trunc_lvl":self.trunc_lvl,
                      "sensitive":self.sensitive,
                      "lmbda":self.lmbda}
            params_path.write_text(json.dumps(params))

            # Run R code
            cmd = ["Rscript",
                   CVINE_SENSITIVE_SCRIPT,
                   data_path,
                   params_path,
                   model_path]

            result = subprocess.run(cmd, capture_output=True)

            if result.returncode != 0:
                raise RuntimeError(
                    f"R estimation failed (exit {result.returncode}).\n"
                    f"stderr:\n{result.stderr}\nstdout:\n{result.stdout}"
                )
              
            self.vc = pv.Vinecop(d=self.d).from_json(model_path.read_text(encoding="utf-8"))

        LOGGER.debug(f'Finished fitting CVineSensitive({self.lmbda})')
        self.trained = True

    def generate_samples(self, nsamples):
        """..."""

    def _read_meta(self, metadata):
            """ Read metadata from metadata file."""
            metadict = {}

            for cdict in metadata['columns']:
                col = cdict['name']
                coltype = cdict['type']

                if coltype == FLOAT or coltype == INTEGER:
                    metadict[col] = {
                        'type': coltype,
                        'min': cdict['min'],
                        'max': cdict['max']
                    }

                elif coltype == CATEGORICAL or coltype == ORDINAL:
                    metadict[col] = {
                        'type': coltype,
                        'categories': cdict['i2s'],
                        'size': len(cdict['i2s'])
                    }

                else:
                    raise ValueError(f'Unknown data type {coltype} for attribute {col}')

            return metadict

In [62]:
s3name = "/uio/hume/student-u82/beritoo/Documents/synthetic_data_release/data/test/test"
data, metadata = load_local_data_as_df(s3name)

In [63]:
gm = CvineSensitive(metadata)
gm.fit(data)

In [64]:
gm.vc

<pyvinecopulib.Vinecop> Vinecop model with 9 variables
tree edge conditioned variables conditioning variables var_types       family rotation       parameters  df   tau 
   1    1                  1, 9                             c, c Independence                                0.00 
   1    2                  2, 9                             c, c      Student        0      -0.03, 2.58 2.0 -0.02 
   1    3                  3, 9                             c, c Independence                                0.00 
   1    4                  4, 9                             c, c        Frank        0            -1.32 1.0 -0.14 
   1    5                  5, 9                             c, c Independence                                0.00 
   1    6                  6, 9                             c, c Independence                                0.00 
   1    7                  7, 9                             c, c Independence                                0.00 
   1    8                